<a href="https://colab.research.google.com/github/ghirr/Smart-Traffic-Flow-Prediction-and-Optimization/blob/main/Part5_Evaluation_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 5 : Évaluation et analyse comparative

**Projet : Smart Traffic Flow Prediction and Optimization**

Ce notebook **ne ré-entraîne rien** : il relit les résultats sauvegardés par les parties 2, 3 et 4 (`results/`) et les compare sur le **jeu de test**, c'est-à-dire la dernière partie de la simulation (15 % des minutes), jamais utilisée pour choisir un modèle ni un hyperparamètre.

**Plan** : 1. chargement · 2. tableau comparatif · 3. les différences sont-elles significatives ? · 4. où les modèles se trompent · 5. coût et variabilité · 6. agent RL · 7. démonstration · 8. limites, pistes et conclusion.

**Choix des métriques**

| Tâche | Métrique principale | Pourquoi |
|---|---|---|
| `etat` (3 classes équilibrées) | **F1 macro**, rappel de la classe « congestionné », AUC | classes équilibrées par construction, mais rater un embouteillage est plus coûteux que rater une circulation fluide, d'où le rappel de cette classe |
| `vitesse` | **MAE** (m/s), RMSE, R² | la MAE est lisible (erreur moyenne en m/s) et peu sensible aux rares pics ; le RMSE pénalise davantage les gros écarts |
| `flux` | **MAE** (véhicules), RMSE, R² | idem ; le flux est un comptage asymétrique avec quelques pics |
| Agent RL | **retard moyen par véhicule** (s), débit, file maximale | mesure directe de l'objectif d'optimisation |

In [ ]:
import os, sys, json, time, warnings, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
SEED = 42
np.random.seed(SEED)

# ---- Données préparées par la partie 1 (dossier data_prepared/ du dépôt) ----
REPO_URL = "https://github.com/ghirr/Smart-Traffic-Flow-Prediction-and-Optimization.git"
BRANCH   = "main"                      # branche du dépôt qui contient data_prepared/ et results/
if not os.path.exists("data_prepared/meta.json") and "google.colab" in sys.modules:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
assert os.path.exists("data_prepared/meta.json"), "Dossier data_prepared/ introuvable : exécutez la partie 1 ou clonez le dépôt."
os.makedirs("results", exist_ok=True)

META   = json.load(open("data_prepared/meta.json"))
FEATS  = META["features"]
ETATS  = META["etats"]                 # ['congestionné', 'ralenti', 'fluide']
SEUILS = META["seuils_etat_m_s"]
SPLITS = ("train", "val", "test")

def charger(suffixe):
    """Retourne les trois jeux (train, val, test) : suffixe = 'raw' (arbres) ou 'scaled' (ANN)."""
    return [pd.read_parquet(f"data_prepared/{s}_{suffixe}.parquet") for s in SPLITS]

# Les trois tâches de prédiction (cibles de la minute suivante)
TASKS = {
    "etat":    dict(target="etat_next_code", kind="clf", nom="État du trafic"),
    "vitesse": dict(target="speed_next",     kind="reg", nom="Vitesse moyenne (m/s)"),
    "flux":    dict(target="n_veh_next",     kind="reg", nom="Flux (nb de véhicules)"),
}

from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             f1_score, recall_score, roc_auc_score, confusion_matrix)

def metriques(kind, y, pred, proba=None):
    """Métriques adaptées à la tâche : régression (MAE, RMSE, R²) ou classification (accuracy, F1 macro, rappels, AUC)."""
    if kind == "reg":
        return {"MAE": mean_absolute_error(y, pred), "RMSE": mean_squared_error(y, pred) ** 0.5, "R2": r2_score(y, pred)}
    rec = recall_score(y, pred, average=None, labels=[0, 1, 2])
    out = {"Accuracy": accuracy_score(y, pred), "F1_macro": f1_score(y, pred, average="macro"),
           "Rappel_congestionné": rec[0], "Rappel_ralenti": rec[1], "Rappel_fluide": rec[2]}
    if proba is not None:
        out["AUC_ovr"] = roc_auc_score(y, proba, multi_class="ovr")
    return out

def perte(kind, y, pred):
    """Critère unique de sélection (à minimiser) : MAE en régression, 1 - F1 macro en classification."""
    return mean_absolute_error(y, pred) if kind == "reg" else 1 - f1_score(y, pred, average="macro")

## 1. Chargement des résultats

Les modèles sont comparés **ligne par ligne** : les prédictions sauvegardées par les parties 2 et 3 portent sur exactement les mêmes lignes de test. On recalcule aussi les références simples (persistance, profil de la cellule) pour savoir si les modèles apportent quelque chose.

In [ ]:
RAW = charger("raw")
TEST, TRAIN = RAW[2], RAW[0]
MODELES_ML = ["RandomForest", "XGBoost", "AdaBoost", "ANN"]
FAMILLE = {"RandomForest": "Bagging", "XGBoost": "Boosting", "AdaBoost": "Boosting", "ANN": "Réseau de neurones"}

PRED, IDX = {}, {}
PROFIL_FLUX = TRAIN.groupby("cell_id")["n_veh_next"].mean()
for t, info in TASKS.items():
    d = TEST.dropna(subset=[info["target"]]); IDX[t] = d.index
    e = pd.read_parquet(f"results/ensemble_preds_{t}.parquet"); a = pd.read_parquet(f"results/ann_preds_{t}.parquet")
    assert np.array_equal(e.y_true.values, a.y_true.values) and len(e) == len(d), "lignes de test non alignées"
    P = {"y_true": e.y_true.values, **{m: e[m].values for m in ["RandomForest", "XGBoost", "AdaBoost"]}, "ANN": a["ANN"].values}
    if t == "vitesse":
        P["Persistance"] = d.speed_roll3.fillna(d.cell_speed_mean).values; P["Profil cellule"] = d.cell_speed_mean.values
    if t == "flux":
        P["Persistance"] = d.n_veh_roll3.values; P["Profil cellule"] = d.cell_id.map(PROFIL_FLUX).fillna(PROFIL_FLUX.mean()).values
    if t == "etat":
        P["Persistance"] = np.digitize(d.speed_roll3.fillna(d.cell_speed_mean), SEUILS); P["Profil cellule"] = np.digitize(d.cell_speed_mean, SEUILS)
    PRED[t] = P
    print(f"{t:8s}: {len(d):6,} lignes de test, {len(P) - 1} modèles / références alignés")

MET_ENS = json.load(open("results/ensemble_metrics.json")); MET_ANN = json.load(open("results/ann_metrics.json")); MET_RL = json.load(open("results/rl_metrics.json"))

## 2. Tableau comparatif sur le test

In [ ]:
PROBA_AUC = {}      # l'AUC vient des métriques sauvegardées (les probabilités n'ont pas été conservées)
rows = []
for t, info in TASKS.items():
    for m, p in PRED[t].items():
        if m == "y_true": continue
        r = metriques(info["kind"], PRED[t]["y_true"], p)
        if info["kind"] == "clf" and m in MODELES_ML:
            src = MET_ANN if m == "ANN" else MET_ENS
            r["AUC_ovr"] = src[t][m]["test"].get("AUC_ovr", np.nan)
        rows.append({"tâche": t, "modèle": m, "famille": FAMILLE.get(m, "Référence"), **r})
TAB = pd.DataFrame(rows)

crit = {"etat": "F1_macro", "vitesse": "MAE", "flux": "MAE"}
for t, info in TASKS.items():
    d = TAB[TAB["tâche"] == t].dropna(axis=1, how="all").drop(columns="tâche")
    d = d.sort_values(crit[t], ascending=(info["kind"] == "reg")).reset_index(drop=True)
    print(f"\n=== {t} : {info['nom']} (trié par {crit[t]}) ===")
    print(d.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
couleur = {"Référence": "lightgray", "Bagging": "seagreen", "Boosting": "steelblue", "Réseau de neurones": "darkorange"}
for a, t in zip(ax, TASKS):
    d = TAB[TAB["tâche"] == t].sort_values(crit[t], ascending=(TASKS[t]["kind"] == "reg"))
    a.barh(d["modèle"], d[crit[t]], color=[couleur[f] for f in d["famille"]])
    a.invert_yaxis(); a.set_title(f"{t} : {crit[t]} (test)" + (" : plus haut = mieux" if TASKS[t]["kind"] == "clf" else " : plus bas = mieux"))
    for i, v in enumerate(d[crit[t]]): a.text(v, i, f" {v:.3f}", va="center", fontsize=8)
    lo, hi = d[crit[t]].min(), d[crit[t]].max(); a.set_xlim(lo - (hi - lo) * 0.6 if TASKS[t]["kind"] == "clf" else 0, hi + (hi - lo) * 0.25)
plt.tight_layout(); plt.show()

## 3. Les différences sont-elles significatives ?

Quand deux modèles ont des scores proches (0,735 contre 0,740, par exemple), on ne peut pas conclure sans mesurer l'**incertitude**. On utilise un **bootstrap par blocs de minutes** : on tire au hasard, avec remise, des minutes entières du test (et pas des lignes isolées, car les cellules d'une même minute sont corrélées entre elles), et on recalcule la différence d'erreur. L'intervalle à 95 % qui ne contient pas 0 indique une différence nette.

Les paires comparées sont fixées **avant** de regarder les résultats : Boosting contre Bagging (XGBoost vs Random Forest), réseau de neurones contre Boosting, et meilleur modèle d'arbres contre persistance.

In [ ]:
def bootstrap_diff(w, err_a, err_b, n=3000, seed=SEED):
    """Différence moyenne d'erreur (a - b) et IC95 % ; négatif = a est meilleur. Rééchantillonnage par minute (bloc)."""
    g = pd.DataFrame({"w": w, "d": err_a - err_b}).groupby("w")["d"].agg(["sum", "count"])
    idx = np.random.default_rng(seed).integers(0, len(g), (n, len(g)))
    m = g["sum"].values[idx].sum(1) / g["count"].values[idx].sum(1)
    return float(g["sum"].sum() / g["count"].sum()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def erreur(t, m):
    y, p = PRED[t]["y_true"], PRED[t][m]
    return np.abs(y - p) if TASKS[t]["kind"] == "reg" else (y != p).astype(float)       # erreur absolue / erreur de classement

PAIRES = [("XGBoost", "RandomForest", "Boosting vs Bagging"), ("ANN", "XGBoost", "ANN vs Boosting"),
          ("XGBoost", "Persistance", "Boosting vs persistance"), ("ANN", "Persistance", "ANN vs persistance")]
lignes = []
for t in TASKS:
    w = TEST.loc[IDX[t], "w_cible"].values
    for a, b, nom in PAIRES:
        diff, lo, hi = bootstrap_diff(w, erreur(t, a), erreur(t, b))
        verdict = f"{a} meilleur" if hi < 0 else (f"{b} meilleur" if lo > 0 else "pas de différence nette")
        lignes.append({"tâche": t, "comparaison": nom, "Δ erreur moyenne": diff, "IC95 bas": lo, "IC95 haut": hi, "verdict": verdict})
print("Δ = erreur de A moins erreur de B (MAE en m/s ou en véhicules ; taux d'erreur de classement pour 'etat').\n")
print(pd.DataFrame(lignes).round(3).to_string(index=False))

## 4. Où les modèles se trompent-ils ?

In [ ]:
# Classification : matrices de confusion normalisées par classe réelle
t = "etat"
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8))
for a, m in zip(ax, ["Persistance", "RandomForest", "XGBoost", "ANN"]):
    sns.heatmap(confusion_matrix(PRED[t]["y_true"], PRED[t][m], normalize="true"), annot=True, fmt=".2f", cmap="Blues", cbar=False,
                xticklabels=["cong.", "ralenti", "fluide"], yticklabels=["cong.", "ralenti", "fluide"], ax=a, vmin=0, vmax=1)
    a.set_title(m); a.set_xlabel("prédit"); a.set_ylabel("réel")
plt.tight_layout(); plt.show()

y = PRED[t]["y_true"]
for m in ["Persistance", "XGBoost", "ANN"]:
    ecart = np.abs(y - PRED[t][m])
    print(f"{m:12s}: erreurs = {100 * (ecart > 0).mean():.1f} % dont {100 * (ecart == 2).mean():.1f} % de confusions extrêmes (congestionné <-> fluide)")

In [ ]:
# Régression : erreur selon la charge de la cellule, et valeurs prédites vs réelles
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
for j, t in enumerate(["vitesse", "flux"]):
    d = TEST.loc[IDX[t]]; y = PRED[t]["y_true"]
    charge = pd.cut(d.n_veh_roll3, [-0.1, 1, 3, 6, 10, 100], labels=["<=1", "1-3", "3-6", "6-10", ">10"])
    err = pd.DataFrame({m: np.abs(y - PRED[t][m]) for m in ["Persistance", "RandomForest", "XGBoost", "ANN"]})
    err.groupby(charge.values, observed=True).mean().plot.bar(ax=ax[j, 0]); ax[j, 0].set_title(f"{t} : MAE selon le flux récent de la cellule")
    ax[j, 0].set_xlabel("véhicules par minute (moyenne des 3 dernières minutes)"); ax[j, 0].tick_params(axis="x", rotation=0)
    for k, m in enumerate(["XGBoost", "ANN"]):
        ax[j, 1 + k].hexbin(y, PRED[t][m], gridsize=40, bins="log", cmap="viridis", mincnt=1)
        lim = [0, max(y.max(), PRED[t][m].max())]; ax[j, 1 + k].plot(lim, lim, "r--", lw=1)
        ax[j, 1 + k].set_title(f"{t} : {m}, prédit vs réel"); ax[j, 1 + k].set_xlabel("réel"); ax[j, 1 + k].set_ylabel("prédit")
plt.tight_layout(); plt.show()

In [ ]:
# Dans le temps (par minute) et dans l'espace (par cellule), pour la vitesse
t = "vitesse"; d = TEST.loc[IDX[t]].copy(); y = PRED[t]["y_true"]
d["err_ref"] = np.abs(y - PRED[t]["Persistance"]); d["err_xgb"] = np.abs(y - PRED[t]["XGBoost"]); d["err_ann"] = np.abs(y - PRED[t]["ANN"])
NX, NY = META["grille"]

fig, ax = plt.subplots(1, 3, figsize=(17, 3.9), gridspec_kw={"width_ratios": [1.4, 1, 1]})
d.groupby("w_cible")[["err_ref", "err_xgb", "err_ann"]].mean().rolling(3, center=True).mean().plot(ax=ax[0])
ax[0].set_title("MAE de la vitesse par minute du test (moyenne glissante 3 min)"); ax[0].set_xlabel("minute de simulation"); ax[0].legend(["persistance", "XGBoost", "ANN"])
for a, col, titre in zip(ax[1:], ["err_ref", "err_xgb"], ["persistance", "XGBoost"]):
    carte = d.groupby("cell_id")[col].mean().reindex(range(NX * NY)).values.reshape(NY, NX)
    sns.heatmap(carte, ax=a, cmap="Reds", vmin=0, vmax=3, cbar=True, cbar_kws={"label": "MAE (m/s)"}); a.invert_yaxis()
    a.set_title(f"MAE par cellule : {titre}"); a.set_xlabel("colonne de la grille"); a.set_ylabel("ligne de la grille")
plt.tight_layout(); plt.show()
print(f"MAE vitesse : persistance {d.err_ref.mean():.3f} | XGBoost {d.err_xgb.mean():.3f} | ANN {d.err_ann.mean():.3f}")
pire = d.groupby("cell_id")["err_xgb"].agg(["mean", "size"]).query("size >= 20").sort_values("mean", ascending=False).head(3)
print("Cellules les plus difficiles pour XGBoost (>= 20 observations) :"); print(pire.round(2).to_string())

## 5. Coût de calcul et variabilité

Un modèle un peu meilleur mais bien plus coûteux ou instable n'est pas toujours le bon choix. On regarde le temps d'entraînement (une exécution sur la machine utilisée) et, pour le réseau de neurones, l'écart-type entre graines aléatoires.

In [ ]:
rows = []
for t in TASKS:
    for m in ["RandomForest", "XGBoost", "AdaBoost"]:
        rows.append({"tâche": t, "modèle": m, "temps d'entraînement (s)": MET_ENS[t][m]["temps_s"]})
    rows.append({"tâche": t, "modèle": "ANN", "temps d'entraînement (s)": MET_ANN[t]["ANN"]["temps_s"]})
cout = pd.DataFrame(rows).pivot(index="modèle", columns="tâche", values="temps d'entraînement (s)")
print("Temps d'entraînement du modèle final (secondes) :"); print(cout.round(1).to_string())

print("\nRéseau de neurones : 3 graines aléatoires, test")
for t in TASKS:
    v = MET_ANN[t]["ANN"]["test_3_graines"][crit[t]]
    print(f"  {t:8s} {crit[t]:9s}: {np.round(v, 3)}  moyenne = {np.mean(v):.3f}, écart-type = {np.std(v):.3f}")

## 6. Agent de Reinforcement Learning (bonus)

L'agent ne prédit pas : il **décide**. Son évaluation est donc différente : on mesure le **retard moyen par véhicule** dans un simulateur de carrefour alimenté par les flux réels du test (voir partie 4), et on le compare à des politiques de régulation de référence.

In [ ]:
reel = pd.DataFrame(MET_RL["reel_test"]).T[["retard_s_par_veh", "ic95", "file_moyenne", "file_max", "debit_veh_h", "changements_h"]]
print("Scénarios réels (test) : 15 créneaux de 15 min x 6 graines")
print(reel.round(2).to_string())
ref = reel.loc[[i for i in reel.index if i.startswith("Temps fixe")][0], "retard_s_par_veh"]
for nom in reel.index:
    print(f"  {nom:32s}: {100 * (reel.loc[nom, 'retard_s_par_veh'] / ref - 1):+6.1f} % de retard par rapport au temps fixe")
print("\nAgents Q-learning (5 graines) sur le test, retard moyen :", np.round(MET_RL["parametres"]["agents_q_test"], 2))

syn = pd.DataFrame(MET_RL["synthetique"]).T
print("\nScénarios synthétiques (retard moyen, s/véh) :"); print(syn.round(1).to_string())

## 7. Démonstration : prévoir l'état du trafic de la minute suivante sur toute la ville

Pour une minute `t` du test, on prend les variables disponibles à la fin de la minute `t`, on prédit l'état de chaque cellule à `t+1` avec **XGBoost**, et on le compare à la réalité.

In [ ]:
t = "etat"; d = TEST.loc[IDX[t]].copy()
d["reel"] = PRED[t]["y_true"]; d["pred"] = PRED[t]["XGBoost"]
NX, NY = META["grille"]

def demo(minute_cible):
    """Carte de l'état réel et de l'état prédit pour la minute `minute_cible` (prédiction faite à la minute précédente)."""
    s = d[d.w_cible == minute_cible]
    fig, ax = plt.subplots(1, 3, figsize=(17, 3.9))
    cmap = plt.matplotlib.colors.ListedColormap(["#c0392b", "#f1c40f", "#27ae60"])
    for a, col, titre in zip(ax[:2], ["reel", "pred"], ["État réel", "État prédit (XGBoost)"]):
        carte = s.set_index("cell_id")[col].reindex(range(NX * NY)).values.reshape(NY, NX)
        im = a.imshow(carte, origin="lower", cmap=cmap, vmin=0, vmax=2, interpolation="nearest"); a.set_title(f"{titre}, minute {minute_cible}")
        a.set_xlabel("colonne de la grille (100 m)"); a.set_ylabel("ligne de la grille (100 m)")
    ok = (s.reel == s.pred); carte = ok.groupby(s.cell_id).first().reindex(range(NX * NY)).astype(float).values.reshape(NY, NX)
    ax[2].imshow(carte, origin="lower", cmap=plt.matplotlib.colors.ListedColormap(["#e74c3c", "#2ecc71"]), vmin=0, vmax=1, interpolation="nearest")
    ax[2].set_title(f"Prédiction correcte : {100 * ok.mean():.0f} % des {len(s)} cellules occupées")
    from matplotlib.patches import Patch
    ax[1].legend(handles=[Patch(color=c, label=l) for c, l in zip(["#c0392b", "#f1c40f", "#27ae60"], ETATS)], loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
    plt.tight_layout(); plt.show()

milieu = int(d.w_cible.median())
demo(milieu)

## 8. Limites, pistes d'amélioration et conclusion

### Ce que l'on retient
**1. Les modèles apprenants battent les références simples, mais de façon modeste.**
Sur le test, par rapport à la persistance (prédire que rien ne change), l'erreur absolue moyenne baisse d'environ **15 %** pour la vitesse (1,504 → 1,280 m/s avec XGBoost) et pour le flux (1,689 → 1,445 véhicules) ; le taux d'erreur de classement de l'état passe de 31,5 % à 27,0 %. Contre la référence « profil de la cellule » (moyenne historique de la cellule), l'avantage est plus faible : −5,7 % sur la vitesse, −10,7 % sur le flux, et +0,017 de F1 macro sur l'état. Une bonne partie de l'information est donc déjà dans l'historique de chaque cellule (la variable `cell_speed_mean` domine le classement des variables de la partie 1) ; les modèles ajoutent surtout la dynamique récente (retards, voisinage).

**2. Les trois familles de modèles sont très proches.**
- **État** : ANN, XGBoost et Random Forest sont à 0,73 de F1 macro ; l'intervalle de confiance de la différence contient 0 (pas de différence nette).
- **Vitesse** : XGBoost (1,280) ≈ Random Forest (1,285) ; le réseau est légèrement moins bon (1,312), et cet écart de 0,03 m/s est statistiquement net mais petit.
- **Flux** : XGBoost est le meilleur (1,445), devant le réseau (1,460) et Random Forest (1,474) ; les différences sont petites mais nettes.
- **AdaBoost** est clairement plus faible, surtout sur la vitesse et le flux (MAE 1,473 et 1,678, c'est-à-dire proche de la persistance) : ses arbres peu profonds sont mal adaptés à des cibles continues bruitées.
- Le seul modèle qui sort un peu du lot est **XGBoost**, et c'est aussi le **plus économique** (3 à 7 s d'entraînement contre 31 à 78 s pour le réseau et 25 à 71 s pour AdaBoost). Si l'on devait en retenir un pour un usage réel, ce serait XGBoost.

**3. Où les modèles se trompent.**
- La classe intermédiaire **« ralenti »** est la plus difficile (rappel de 0,62 à 0,64) ; les confusions extrêmes (congestionné ↔ fluide) ne représentent que 2,5 % des échantillons. Comme les classes sont définies par des seuils sur une vitesse continue, une erreur de classement est souvent une erreur de quelques dixièmes de m/s de part et d'autre d'un seuil.
- Les erreurs ne sont pas uniformes dans l'espace : elles se concentrent sur quelques cellules (par exemple 12, 167 et 46, avec une MAE moyenne de 2,9 à 3,7 m/s) et sur certains bords de la grille. Elles évoluent aussi dans le temps de façon quasi identique pour tous les modèles, ce qui montre qu'une partie de l'erreur vient de variations du trafic qu'aucun de ces modèles ne voit venir.
- Sur la carte de démonstration (minute 364), 72 % des cellules occupées sont prédites correctement, ce qui est cohérent avec l'exactitude globale d'environ 73 %.

**4. Le bonus RL.**
L'agent Q-learning ramène le retard moyen de 30,1 s (temps fixe optimisé) à 11,5 s (−62 %), et fait légèrement mieux que l'heuristique « plus longue file » (11,9 s ; différence appariée de −0,34 s, intervalle de confiance excluant 0). L'avantage est surtout net en forte charge (−16 %). Ces retards sont des valeurs de simulation, utiles pour comparer des politiques entre elles, pas des mesures du réseau réel.

### Limites
- **Un seul jeu de données et un seul découpage temporel** (70 % / 15 % / 15 %, avec 394 fenêtres d'une minute au total). Les intervalles de confiance sont calculés par bootstrap par blocs de minutes et mesurent l'incertitude d'échantillonnage sur cette période de test, pas la variabilité entre journées de trafic différentes.
- **Horizon d'une minute** : on prédit la fenêtre suivante. Un horizon plus long serait plus utile pour agir sur le trafic, mais plus difficile.
- **Cibles dérivées** : l'état est un découpage par seuils (quantiles du jeu d'entraînement) de la vitesse ; un autre découpage donnerait d'autres scores.
- **Cellules peu observées** : en dessous de quelques véhicules par fenêtre, la vitesse moyenne est bruitée, ce qui borne la précision atteignable.
- **Réglage limité par le temps de calcul** : 6 essais pour le réseau, 25 pour XGBoost, 4 pour AdaBoost. Des écarts de moins de 2 % ne doivent pas être sur-interprétés.
- **RL** : environnement simplifié (carrefour à 2 phases, hypothèses de saturation et de temps perdu choisies par nous), apprentissage tabulaire à états discrétisés.

### Pistes d'amélioration
- Élargir le voisinage spatial (au-delà des cellules adjacentes déjà utilisées) ou utiliser un modèle qui exploite la grille : réseau convolutif, ConvLSTM ou réseau de graphes sur le réseau routier.
- Ajouter des variables de contexte (heure, jour, événements) si le jeu de données le permet, et valider sur plusieurs périodes (validation croisée temporelle).
- Prédire plusieurs horizons (1, 5, 15 minutes) et fournir un intervalle de prédiction (régression quantile) plutôt qu'une seule valeur.
- Pour le RL : calibrer l'environnement sur le simulateur d'origine, passer à un **DQN** pour éviter la discrétisation, et étendre le contrôle à plusieurs carrefours voisins en reliant le prédicteur de flux à l'agent.

### Conclusion
Le pipeline complet (préparation sans fuite de données, ensembles, réseau de neurones, apprentissage par renforcement) donne des résultats cohérents entre eux : l'information utile est concentrée dans l'historique local de la circulation, les modèles les plus simples à régler (XGBoost, Random Forest) suffisent, et le réseau de neurones ne les dépasse pas sur ces données tabulaires. L'intérêt pratique est surtout de **combiner** la prévision (où le trafic va se dégrader) avec la **décision** (comment adapter les feux), ce que l'agent de RL illustre sur un carrefour simulé.